In [9]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

In [10]:
load_dotenv()

True

In [11]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [12]:
class JokeState(TypedDict):
    joke: str
    topic: str
    explanation: str

In [13]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [14]:
graph = StateGraph(JokeState)
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [15]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'joke': "Why did the pizza get a bad grade in geometry?\n\nBecause it's round, comes in a square box, and you eat it in triangles!",
 'topic': 'pizza',
 'explanation': 'This joke is funny because it plays on the **irony and absurdity** of applying human academic standards to an inanimate object (pizza), while simultaneously highlighting how perfectly the pizza embodies various geometric concepts.\n\nHere\'s a breakdown:\n\n1.  **The Setup (Geometry Class):** Geometry is the branch of mathematics concerned with the properties and relations of points, lines, surfaces, solids, and higher dimensional analogs. In simple terms, it\'s the study of shapes, sizes, positions, and properties of space.\n\n2.  **The Punchline (The Pizza\'s "Problems"):**\n    *   **"It\'s round":** A circle is one of the most fundamental shapes studied in geometry. Its circumference, radius, diameter, and area are all key geometric concepts.\n    *   **"Comes in a square box":** A square (or a rectangular prism fo

In [16]:
workflow.get_state(config1)

StateSnapshot(values={'joke': "Why did the pizza get a bad grade in geometry?\n\nBecause it's round, comes in a square box, and you eat it in triangles!", 'topic': 'pizza', 'explanation': 'This joke is funny because it plays on the **irony and absurdity** of applying human academic standards to an inanimate object (pizza), while simultaneously highlighting how perfectly the pizza embodies various geometric concepts.\n\nHere\'s a breakdown:\n\n1.  **The Setup (Geometry Class):** Geometry is the branch of mathematics concerned with the properties and relations of points, lines, surfaces, solids, and higher dimensional analogs. In simple terms, it\'s the study of shapes, sizes, positions, and properties of space.\n\n2.  **The Punchline (The Pizza\'s "Problems"):**\n    *   **"It\'s round":** A circle is one of the most fundamental shapes studied in geometry. Its circumference, radius, diameter, and area are all key geometric concepts.\n    *   **"Comes in a square box":** A square (or a r

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'joke': "Why did the pizza get a bad grade in geometry?\n\nBecause it's round, comes in a square box, and you eat it in triangles!", 'topic': 'pizza', 'explanation': 'This joke is funny because it plays on the **irony and absurdity** of applying human academic standards to an inanimate object (pizza), while simultaneously highlighting how perfectly the pizza embodies various geometric concepts.\n\nHere\'s a breakdown:\n\n1.  **The Setup (Geometry Class):** Geometry is the branch of mathematics concerned with the properties and relations of points, lines, surfaces, solids, and higher dimensional analogs. In simple terms, it\'s the study of shapes, sizes, positions, and properties of space.\n\n2.  **The Punchline (The Pizza\'s "Problems"):**\n    *   **"It\'s round":** A circle is one of the most fundamental shapes studied in geometry. Its circumference, radius, diameter, and area are all key geometric concepts.\n    *   **"Comes in a square box":** A square (or a 

In [18]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'joke': 'What do you call a fake noodle?\nAn **impasta**!',
 'topic': 'pasta',
 'explanation': 'This joke is a **pun**, which is a form of wordplay that exploits multiple meanings of a word or words that sound similar but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Fake noodle"**: This is the setup. Noodles are a type of pasta.\n2.  **"Impasta"**: This is the punchline, and it\'s a blend of two ideas:\n    *   **"Pasta"**: This is the obvious connection to the "noodle" in the question.\n    *   **"Impostor"**: This is the word it sounds almost exactly like. An "impostor" is a person who pretends to be someone else in order to deceive others, or something that is not what it purports to be; a fake.\n\nSo, the joke cleverly combines the idea of something being "fake" (like an impostor) with the food "pasta" (a noodle), creating "impasta" – a silly and fitting name for a fake noodle. The humor comes from the unexpected but perfectly fitting wordplay.'}

In [19]:
workflow.get_state(config2)

StateSnapshot(values={'joke': 'What do you call a fake noodle?\nAn **impasta**!', 'topic': 'pasta', 'explanation': 'This joke is a **pun**, which is a form of wordplay that exploits multiple meanings of a word or words that sound similar but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Fake noodle"**: This is the setup. Noodles are a type of pasta.\n2.  **"Impasta"**: This is the punchline, and it\'s a blend of two ideas:\n    *   **"Pasta"**: This is the obvious connection to the "noodle" in the question.\n    *   **"Impostor"**: This is the word it sounds almost exactly like. An "impostor" is a person who pretends to be someone else in order to deceive others, or something that is not what it purports to be; a fake.\n\nSo, the joke cleverly combines the idea of something being "fake" (like an impostor) with the food "pasta" (a noodle), creating "impasta" – a silly and fitting name for a fake noodle. The humor comes from the unexpected but perfectly fitting wordplay.'}

In [20]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'joke': 'What do you call a fake noodle?\nAn **impasta**!', 'topic': 'pasta', 'explanation': 'This joke is a **pun**, which is a form of wordplay that exploits multiple meanings of a word or words that sound similar but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Fake noodle"**: This is the setup. Noodles are a type of pasta.\n2.  **"Impasta"**: This is the punchline, and it\'s a blend of two ideas:\n    *   **"Pasta"**: This is the obvious connection to the "noodle" in the question.\n    *   **"Impostor"**: This is the word it sounds almost exactly like. An "impostor" is a person who pretends to be someone else in order to deceive others, or something that is not what it purports to be; a fake.\n\nSo, the joke cleverly combines the idea of something being "fake" (like an impostor) with the food "pasta" (a noodle), creating "impasta" – a silly and fitting name for a fake noodle. The humor comes from the unexpected but perfectly fitting wordplay.'

# Time Travel

In [27]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1573ff-84cc-6d18-bfff-bd88571aa17c"}})

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1573ff-84cc-6d18-bfff-bd88571aa17c'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-05-24T07:12:48.553500+00:00', parent_config=None, tasks=(PregelTask(id='1d263d79-ff78-fe51-2c91-489815c49ade', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'topic': 'pizza'}),), interrupts=())

In [28]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1573ff-84cc-6d18-bfff-bd88571aa17c"}})

{'joke': 'Why did the pizza get a job?\n\nBecause it needed to make some **dough**!',
 'topic': 'pizza',
 'explanation': 'This joke is a classic example of a **pun**, which is a play on words that uses a word or phrase that has two different meanings, or two words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Dough" (Meaning 1 - Literal):** In the context of pizza, "dough" is the main ingredient – the mixture of flour, water, yeast, etc., that forms the crust. A pizza literally *is* dough.\n\n2.  **"Dough" (Meaning 2 - Slang):** "Dough" is a common slang term for **money**. When people get a job, they do it to "make some dough" (i.e., earn money).\n\nThe humor comes from the joke playing on both meanings simultaneously:\n\n*   When you hear "Why did the pizza get a job?", your brain expects a human-like reason, and "needed to make some **dough**" immediately makes you think of **money**.\n*   But then you realize that a pizza literally *is* made of *

In [29]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'Why did the pizza get a job?\n\nBecause it needed to make some **dough**!', 'topic': 'pizza', 'explanation': 'This joke is a classic example of a **pun**, which is a play on words that uses a word or phrase that has two different meanings, or two words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Dough" (Meaning 1 - Literal):** In the context of pizza, "dough" is the main ingredient – the mixture of flour, water, yeast, etc., that forms the crust. A pizza literally *is* dough.\n\n2.  **"Dough" (Meaning 2 - Slang):** "Dough" is a common slang term for **money**. When people get a job, they do it to "make some dough" (i.e., earn money).\n\nThe humor comes from the joke playing on both meanings simultaneously:\n\n*   When you hear "Why did the pizza get a job?", your brain expects a human-like reason, and "needed to make some **dough**" immediately makes you think of **money**.\n*   But then you realize that a pizza lite

In [30]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1573ff-84e7-6650-8000-f05b1901bd97", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f157405-14c8-6559-8001-f20ea21fd142'}}

In [31]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f157405-14c8-6559-8001-f20ea21fd142'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-05-24T07:15:17.868885+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1573ff-84e7-6650-8000-f05b1901bd97'}}, tasks=(PregelTask(id='1626d337-2cfc-1a5d-cd2e-58a1c776847b', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'joke': 'Why did the pizza get a job?\n\nBecause it needed to make some **dough**!', 'topic': 'pizza', 'explanation': 'This joke is a classic example of a **pun**, which is a play on words that uses a word or phrase that has two different meanings, or two words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Dough" (Meani

In [32]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f157405-14c8-6559-8001-f20ea21fd142"}})

{'joke': 'Why did the samosa look so tired?\n\nBecause it had been *fried* all day!',
 'topic': 'samosa',
 'explanation': 'This joke is a classic pun that plays on words that sound similar but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **The Setup:** "Why did the samosa look so tired?"\n    *   This immediately makes you think of a person being tired, which is funny because a samosa (a food item) can\'t actually be tired. This is called anthropomorphism – giving human qualities to inanimate objects.\n\n2.  **The Punchline:** "Because it had been *fried* all day!"\n    *   **Literal Meaning (for a samosa):** Samosas are cooked by being deep-fried in oil. So, a samosa literally *is* fried. "All day" emphasizes a long, continuous process of being cooked.\n    *   **The Pun:** The word "fried" sounds almost exactly like "tried."\n    *   **The Connection:** When a *person* has "tried all day" (meaning they\'ve put in a lot of effort or worked hard), they would naturally be "

In [33]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'Why did the samosa look so tired?\n\nBecause it had been *fried* all day!', 'topic': 'samosa', 'explanation': 'This joke is a classic pun that plays on words that sound similar but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **The Setup:** "Why did the samosa look so tired?"\n    *   This immediately makes you think of a person being tired, which is funny because a samosa (a food item) can\'t actually be tired. This is called anthropomorphism – giving human qualities to inanimate objects.\n\n2.  **The Punchline:** "Because it had been *fried* all day!"\n    *   **Literal Meaning (for a samosa):** Samosas are cooked by being deep-fried in oil. So, a samosa literally *is* fried. "All day" emphasizes a long, continuous process of being cooked.\n    *   **The Pun:** The word "fried" sounds almost exactly like "tried."\n    *   **The Connection:** When a *person* has "tried all day" (meaning they\'ve put in a lot of effort or worked hard), they 

# Fault Tolerance

In [2]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [4]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [5]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [6]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


EmptyInputError: Received no input for __start__

In [7]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[]